# Bechdel Test Prediction — Comprehensive ML Experiment Suite

**University Group Project — ML Foundations Course**

---

## Research Questions

1. Is the model learning **meaningful representation-related patterns** from pre-release movie metadata?
2. Or is it relying on **shortcut / confounding features** (genre stereotypes, popularity, temporal trends, enrichment artifacts)?

## Notebook Structure
| Section | Content |
|---------|--------|
| 1 | Setup & Imports |
| 2 | Load Data |
| 3 | Feature Engineering & Fixed Train/Test Split |
| 4 | Helper Functions |
| 5 | Reusable Experiment Runner |
| 6 | Exploratory Data Analysis |
| 7 | Model Selection — Baseline (all models) |
| 8 | Imputation Strategy Experiments |
| 9 | Missingness-as-Signal Experiment |
| 10 | Ablation Experiments |
| 11 | Results Comparison Tables |
| 12 | Visualisations & Interpretation |
| 13 | Final Conclusions |

**Design rules (enforced throughout):**
- Train/test split is fixed across every experiment (`seed=42`, `stratify=y`, 80/20).
- Every experiment builds a **fresh, unfitted pipeline** — no object reuse across runs.
- Preprocessing (imputation, scaling, OHE, TF-IDF) is **fitted only on training folds** during CV.
- The held-out test set is used only for final evaluation, never for model or hyperparameter selection.

## 1. Setup & Imports

In [79]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath('../src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, learning_curve as lc
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance as perm_imp

# ── Optional heavy dependencies ───────────────────────────────────────────
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('SHAP not installed; SHAP plots will be skipped.  pip install shap')

try:
    import missingno as msno
    HAS_MSNO = True
except ImportError:
    HAS_MSNO = False

# ── Project source modules ────────────────────────────────────────────────
from data_loader import load_data
from preprocessing import (
    engineer_features, get_feature_lists,
    NUMERIC_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES,
    GENRE_LIST, NUMERIC_BASE, NUMERIC_TMDB,
    _DensePlotUnion, ColumnSelector
)
from models import get_classifiers, HAS_XGB

# ── Global experiment constants ───────────────────────────────────────────
RANDOM_STATE = 42
CV_SPLITS    = 5
TEST_SIZE    = 0.20
SCORING      = 'roc_auc'

# ── Plotting style ────────────────────────────────────────────────────────
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('colorblind')
COLORS = sns.color_palette('colorblind', 12)

print('Environment ready.')
print(f'  SHAP available:      {HAS_SHAP}')
print(f'  missingno available: {HAS_MSNO}')
print(f'  XGBoost available:   {HAS_XGB}')


Environment ready.
  SHAP available:      True
  missingno available: False
  XGBoost available:   True


## 2. Load Data

`movies_complete.csv` is the fully merged dataset (Bechdel/IMDb + TMDb enrichment + Wikidata supplement). Films without a Bechdel rating are automatically excluded by the loader.

**Expected missingness:** budget, revenue, and TMDb popularity are missing for ~50–78% of films depending on the dataset version — this is by design, not a data quality issue. The enrichment comes from a separate source with only partial overlap.

In [80]:
DATA_DIR      = os.path.abspath('../data')
COMPLETE_PATH = os.path.join(DATA_DIR, 'movies_complete.csv')
BECHDEL_PATH  = os.path.join(DATA_DIR, 'Bechdel_IMDB_Merge0524 copy.csv')
TMDB_PATH     = os.path.join(DATA_DIR, 'movies_enriched_5k.csv')

df_raw = load_data(
    bechdel_path=BECHDEL_PATH,
    tmdb_path=TMDB_PATH,
    complete_path=COMPLETE_PATH if os.path.exists(COMPLETE_PATH) else None,
)

print(f'\nDataset shape: {df_raw.shape}')
vc = df_raw['bechdel_pass'].value_counts()
print(f'Target distribution: Pass={vc.get(1,0):,} ({vc.get(1,0)/len(df_raw):.1%})  '
      f'Fail={vc.get(0,0):,} ({vc.get(0,0)/len(df_raw):.1%})')
print('\nMissingness rates (%):')
miss = df_raw.isnull().mean().mul(100).round(1)
print(miss[miss > 0].to_string())
df_raw.head(3)

[data_loader] Dropped 1,618 unlabelled movies (no Bechdel rating). 9,718 labelled films retained.
[data_loader] Loaded 9,718 films from complete dataset.

Dataset shape: (9718, 15)
Target distribution: Pass=5,602 (57.6%)  Fail=4,116 (42.4%)

Missingness rates (%):
runtime                 0.1
budget                 55.2
revenue                56.7
tmdb_popularity        65.2
cast_size               1.2
has_female_director     2.2


,imdb_id,title,year,runtime,imdb_rating,num_votes,bechdel_score,bechdel_pass,genres,budget,revenue,tmdb_popularity,cast_size,has_female_director,plot
0,tt0000009,Miss Jerry,1894.0,45.0,5.4,212.0,0.0,0,Romance,NaN,NaN,NaN,0.0,0.0,1894 film directed by Alexander Black
1,tt0000574,"Story of the Kelly Gang, The",1906.0,70.0,6.0,903.0,1.0,0,Action|Adventure|Biography,NaN,NaN,NaN,1.0,0.0,1906 film
2,tt0002101,Cleopatra,1912.0,100.0,5.1,622.0,2.0,0,Drama|History,NaN,NaN,NaN,3.0,0.0,1912 film by Charles L. Gaskill


## 3. Feature Engineering & Fixed Train/Test Split

All feature engineering is **row-local and deterministic**: log-transforms, decade bucketing, and genre one-hot flags depend only on a row's own values, so applying them before the split introduces zero leakage.

Raw columns (`runtime`, `num_votes`, `budget`, `revenue`, `cast_size`, `genres`) are replaced by their engineered versions. The raw columns and identifiers are dropped from `X`; the `plot` column is kept so TF-IDF experiments can use it inside the pipeline.

> **The train/test split is fixed for the entire notebook.** All experiments use exactly the same split.

In [81]:
# Row-local engineering — safe to do before split
df = engineer_features(df_raw)

TARGET = 'bechdel_pass'

# Columns to exclude from X:
#   identifiers, the raw columns (replaced by engineered versions), and the target
DROP_COLS = [
    'imdb_id', 'title',
    'bechdel_score', 'bechdel_pass',  # target leakage
    'runtime', 'num_votes',           # replaced by log_runtime, log_num_votes
    'budget', 'revenue', 'cast_size', # replaced by log_* versions
    'genres',                         # replaced by genre_* one-hots
    # 'plot' intentionally KEPT for optional TF-IDF inside the pipeline
]

X = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print(f'Train pass rate: {y_train.mean():.3f}  |  Test pass rate: {y_test.mean():.3f}')
print(f'\nAll feature columns ({len(X_train.columns)}):')
print(list(X_train.columns))

Train: 7,774 rows  |  Test: 1,944 rows
Train pass rate: 0.576  |  Test pass rate: 0.577

All feature columns (31):
['year', 'imdb_rating', 'tmdb_popularity', 'has_female_director', 'plot', 'log_runtime', 'log_num_votes', 'log_budget', 'log_revenue', 'log_cast_size', 'decade', 'genre_action', 'genre_adventure', 'genre_animation', 'genre_biography', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_fantasy', 'genre_history', 'genre_horror', 'genre_music', 'genre_musical', 'genre_mystery', 'genre_romance', 'genre_sci_fi', 'genre_sport', 'genre_thriller', 'genre_war', 'genre_western']


## 4. Helper Functions

### 4a. Missingness Indicator Creator
For the **missingness-as-signal experiment** we add binary flags (`budget_missing`, `revenue_missing`, etc.) encoding whether each TMDb-derived feature was absent. These flags are computed from the data **before** any pipeline fitting, so they are safe to include as features.

### 4b. Feature Group Definitions
We define which columns belong to each feature group for ablation experiments. All definitions are derived from `X_train.columns` so they reflect exactly what is available after the split.

### 4c. Fresh Pipeline Builder
`build_experiment_pipeline()` constructs a **fully new, unfitted sklearn Pipeline** every time it is called. The imputation strategy, TF-IDF inclusion, and feature lists are all configurable per call.

In [82]:
# ── 4a. Missingness indicators ────────────────────────────────────────────
ENRICHMENT_COLS_MAP = {
    'log_budget':      'budget_missing',
    'log_revenue':     'revenue_missing',
    'tmdb_popularity': 'popularity_missing',
    'log_cast_size':   'cast_size_missing',
}
PLOT_MISSING_COL = 'plot_missing'


def add_missingness_indicators(X: pd.DataFrame) -> pd.DataFrame:
    """Add binary missingness indicator columns for enrichment-derived features."""
    X = X.copy()
    for feat_col, indicator_name in ENRICHMENT_COLS_MAP.items():
        X[indicator_name] = X[feat_col].isna().astype(int) if feat_col in X.columns else 0
    # plot_missing: empty string or NaN means no plot was available
    if 'plot' in X.columns:
        X[PLOT_MISSING_COL] = X['plot'].fillna('').eq('').astype(int)
    else:
        X[PLOT_MISSING_COL] = 0
    return X


# ── 4b. Feature group definitions for ablations ───────────────────────────
GENRE_FEATURES = [c for c in X_train.columns if c.startswith('genre_')]

POPULARITY_FEATURES = [
    c for c in ['imdb_rating', 'log_num_votes', 'log_budget', 'log_revenue', 'tmdb_popularity']
    if c in X_train.columns
]

TEMPORAL_FEATURES = [c for c in ['year', 'decade'] if c in X_train.columns]

TMDB_ENRICHMENT_FEATURES = [
    c for c in ['log_budget', 'log_revenue', 'tmdb_popularity', 'log_cast_size', 'has_female_director']
    if c in X_train.columns
]

print('Feature group sizes:')
print(f'  Genre features:      {len(GENRE_FEATURES):>3}  {GENRE_FEATURES[:3]}...')
print(f'  Popularity features: {len(POPULARITY_FEATURES):>3}  {POPULARITY_FEATURES}')
print(f'  Temporal features:   {len(TEMPORAL_FEATURES):>3}  {TEMPORAL_FEATURES}')
print(f'  TMDb enrichment:     {len(TMDB_ENRICHMENT_FEATURES):>3}  {TMDB_ENRICHMENT_FEATURES}')

Feature group sizes:
  Genre features:       20  ['genre_action', 'genre_adventure', 'genre_animation']...
  Popularity features:   5  ['imdb_rating', 'log_num_votes', 'log_budget', 'log_revenue', 'tmdb_popularity']
  Temporal features:     2  ['year', 'decade']
  TMDb enrichment:       5  ['log_budget', 'log_revenue', 'tmdb_popularity', 'log_cast_size', 'has_female_director']


In [83]:
# ── 4c. Fresh pipeline builder ────────────────────────────────────────────
def build_experiment_pipeline(
    clf,
    numeric_features,
    categorical_features,
    binary_features,
    imputation_strategy='median',
    use_plot_tfidf=False,
):
    """
    Build a fresh, unfitted sklearn Pipeline.
    Every call creates independent estimator objects — no shared state between experiments.
    """
    valid_strategies = ('mean', 'median', 'most_frequent')
    num_strategy = imputation_strategy if imputation_strategy in valid_strategies else 'median'

    dense_ct = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('impute', SimpleImputer(strategy=num_strategy)),
                ('scale',  StandardScaler()),
            ]), numeric_features),
            ('cat', Pipeline([
                ('impute', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore',
                                         sparse_output=False, drop='first')),
            ]), categorical_features),
            ('bin', SimpleImputer(strategy='constant', fill_value=0), binary_features),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )

    if use_plot_tfidf:
        tfidf_pipe = Pipeline([
            ('sel',   ColumnSelector('plot')),
            ('tfidf', TfidfVectorizer(
                max_features=150, sublinear_tf=True,
                min_df=3, ngram_range=(1, 2), stop_words='english',
            )),
        ])
        preprocessor = _DensePlotUnion(dense_ct, tfidf_pipe)
    else:
        preprocessor = dense_ct

    return Pipeline([
        ('preprocess', preprocessor),
        ('clf', clone(clf)),
    ])


print('Pipeline builder defined.')

Pipeline builder defined.
